# Retrival

In diesem Schritt geht es um das Query an die Datenbank um uns eine reihe von ähnlichen Chunks zu erhalten.  Alle Testfragen werde nacheinander an die DB gesendet... zuvor natürlich das Embedding der Frage.

Um die Retrivals zu optimieren, wird ein Crossencoder verwendet. Er vergleicht die Ergebnisse direkt mit dem Query und kann so besser die Ähnlichkeit ermitteln. Das Verwendete Modell soll gut mit deutschen und kurzen Texten sehr gut zurecht kommen.

* Rerankingmodel: BAAI/bge-reranker-v2-m3

Evaluiert wird per

* Recall@k: Wurde der korrekte Chunk gefunden?
* Mean Reciprocal Rank (MRR): Auf welcher Postition wurde der korrekte Chunk durchschnittlich gefunden?
* Precision@k: Wie relevant sind die gefundenen Chunks?

Für die Fragen mit einem Chunk als Antwort wird hier Recall@k und MRR verwendet. Precision@k ist hingegen für jene Fragen wichtig, die mehr als nur einen Chunk zur Beantwortung benötigen.

Schlechte Retrivales werden nochmal untersucht um zu prüfen, wo es Verbesserungspotential gibt.

In [9]:
import json
import chromadb
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

embedding_model = SentenceTransformer('deepset/gbert-large')
cencoder_model = CrossEncoder('BAAI/bge-reranker-v2-m3')
db_client = chromadb.PersistentClient(path="../data/vector_store")

db_collection = db_client.get_collection('ProduktRAG')

No sentence-transformers model found with name deepset/gbert-large. Creating a new one with mean pooling.


In [10]:
# Fragen stellen, Retrival erhalten
with open('../data/tests/specs_question.json', 'r') as f:
    specs_question = json.load(f)

retrivals = []

for quest in tqdm(specs_question, total=len(specs_question)):
    embedding = embedding_model.encode(quest['question']).tolist()

    result = db_collection.query(query_embeddings=[embedding], n_results=20)
    retrivals.append(result)

print(f"Anzahl Queries: {len(specs_question)}")
print(f"Anzahl Results: {len(retrivals)}")

100%|██████████| 64/64 [00:29<00:00,  2.16it/s]

Anzahl Queries: 64
Anzahl Results: 64


## Evaluation

In [ ]:
recall_at_1, recall_at_3, recall_at_5, recall_at_10, no_recall = [], [], [], [], []
mrr_scores = []
quest_list = []
badies = []

for i in range(len(specs_question)):
    expected_id = specs_question[i]['chunk_id']
    retrieved_ids = retrivals[i]['ids'][0]

    # Recall
    recall_at_1.append(1 if expected_id in retrieved_ids[:1] else 0)
    recall_at_3.append(1 if expected_id in retrieved_ids[:3] else 0)
    recall_at_5.append(1 if expected_id in retrieved_ids[:5] else 0)
    recall_at_10.append(1 if expected_id in retrieved_ids[:10] else 0)
    no_recall.append(1 if expected_id not in retrieved_ids else 0)

    # MRR
    if expected_id in retrieved_ids:
        position = retrieved_ids.index(expected_id) + 1
        mrr_scores.append(1.0 / position)
    else:
        mrr_scores.append(0.0)
    
    # Get bad retrivals
    if recall_at_5[i] == 0:
        badies.append({
            'query': specs_question[i]['question'],
            'answers': retrivals[i]['documents']
        })

print(f"Recall@1:  {sum(recall_at_1) / len(recall_at_1):.2%}")
print(f"Recall@3:  {sum(recall_at_3) / len(recall_at_3):.2%}")
print(f"Recall@5:  {sum(recall_at_5) / len(recall_at_5):.2%}")
print(f"Recall@10: {sum(recall_at_10) / len(recall_at_10):.2%}")
print()
print(f"No Recall: {sum(no_recall) / len(no_recall):.2%}")
print()
print(f"MRR:       {sum(mrr_scores) / len(mrr_scores):.3f}")
print()
print(json.dumps(badies, indent=2, ensure_ascii=False))


Recall@1:  18.75%
Recall@3:  34.38%
Recall@5:  39.06%
Recall@10: 48.44%
Recall@20: 60.94%
No Recall: 39.06%

MRR:       0.294

[
  {
    "query": "Wie hoch ist der Stromverbrauch des Kirsch LABEX-720 PRO-ACTIVE in 24 Stunden?",
    "answers": [
      [
        "Kirsch LABEX-720 PRO-ACTIVE: Leistungsaufnahme 237 Watt",
        "Kirsch LABEX-520 PRO-ACTIVE: Leistungsaufnahme 234 Watt",
        "Kirsch LABO-720 PRO-ACTIVE: Leistungsaufnahme 250 Watt",
        "Kirsch FROSTER BL-530 PRO-ACTIVE: Leistungsaufnahme 740 Watt",
        "Kirsch FROSTER BL-730 PRO-ACTIVE: Leistungsaufnahme 790 Watt",
        "Kirsch BL-720 PRO-ACTIVE: Leistungsaufnahme 250 Watt",
        "Kirsch LABO-520 PRO-ACTIVE: Leistungsaufnahme 250 Watt",
        "Kirsch LABEX-105 PRO-ACTIVE: Leistungsaufnahme 86 Watt",
        "Kirsch LABO-100 PRO-ACTIVE: Leistungsaufnahme 76 Watt",
        "Kirsch LABEX-720 PRO-ACTIVE: Normalverbrauch 1,23 kWh/24 h",
        "Kirsch LABO-288 PRO-ACTIVE: Leistungsaufnahme 88 Watt",
       

## Reranking

In [ ]:
rerankings = []

for i, quest in tqdm(enumerate(specs_question), total=len(specs_question)):

    question = quest['question']
    answer_ids = retrivals[i]['ids'][0]
    answer_texts = retrivals[i]['documents'][0]

    # Crossencoding durchführen
    pairs = [[question, candidate] for candidate in answer_texts]
    scores = cencoder_model.predict(pairs)

    # Reranking durchführen
    results_random = list(zip(answer_ids, answer_texts, scores))
    results_sorted = sorted(results_random, key=lambda x: x[2], reverse=True)

    rerankings.append({
        'ids': [[chunk_id for chunk_id, text, score in results_sorted]]
    })

 19%|█▉        | 12/64 [01:16<05:44,  6.63s/it]

## Evaluation Reranking

In [18]:
rrecall_at_1, rrecall_at_3, rrecall_at_5, rrecall_at_10 = [], [], [], []
rmrr_scores = []
quest_list = []

for i in range(len(specs_question)):
    expected_id = specs_question[i]['chunk_id']
    retrieved_ids = rerankings[i]['ids'][0]

    # Recall
    rrecall_at_1.append(1 if expected_id in retrieved_ids[:1] else 0)
    rrecall_at_3.append(1 if expected_id in retrieved_ids[:3] else 0)
    rrecall_at_5.append(1 if expected_id in retrieved_ids[:5] else 0)
    rrecall_at_10.append(1 if expected_id in retrieved_ids[:10] else 0)

    # MRR
    if expected_id in retrieved_ids:
        position = retrieved_ids.index(expected_id) + 1
        rmrr_scores.append(1.0 / position)
    else:
        rmrr_scores.append(0.0)

print(f"Recall@1:  {sum(rrecall_at_1) / len(rrecall_at_1):.2%}")
print(f"Recall@3:  {sum(rrecall_at_3) / len(rrecall_at_3):.2%}")
print(f"Recall@5:  {sum(rrecall_at_5) / len(rrecall_at_5):.2%}")
print(f"Recall@10: {sum(rrecall_at_10) / len(rrecall_at_10):.2%}")
print()
print(f"MRR:       {sum(rmrr_scores) / len(rmrr_scores):.3f}")
print()
# print(json.dumps(badies, indent=2, ensure_ascii=False))


Recall@1:  0.00%
Recall@3:  0.00%
Recall@5:  0.00%
Recall@10: 0.00%

MRR:       0.000

